In [4]:
import pandas as pd

policy_data = pd.read_csv("../data/raw/policy_data.csv")

policy_data

,Policy_ID,Age,Sum_Assured,Annual_Premium,Policy_Term
0,1001,35,1000000,12000.0,20
1,1002,-5,1500000,18000.0,15
2,1003,29,750000,NaN,25
3,1004,51,2000000,26000.0,10
4,1005,138,1250000,15000.0,18


In [34]:
policy_data.info

<bound method DataFrame.info of    Policy_ID  Age  Sum_Assured  Annual_Premium  Policy_Term
0       1001   35      1000000         12000.0           20
1       1002   -5      1500000         18000.0           15
2       1003   29       750000             NaN           25
3       1004   51      2000000         26000.0           10
4       1005  138      1250000         15000.0           18>

In [7]:
clean_policy_data = policy_data[
    (policy_data["Age"]>=0)&
    (policy_data["Age"]<100)
    ].copy()

clean_policy_data

,Policy_ID,Age,Sum_Assured,Annual_Premium,Policy_Term
0,1001,35,1000000,12000.0,20
2,1003,29,750000,NaN,25
3,1004,51,2000000,26000.0,10


In [8]:
clean_policy_data.info

<bound method DataFrame.info of    Policy_ID  Age  Sum_Assured  Annual_Premium  Policy_Term
0       1001   35      1000000         12000.0           20
2       1003   29       750000             NaN           25
3       1004   51      2000000         26000.0           10>

In [9]:
def clean_policy_data(df):
    clean_df = df[
        (df["Age"]>=0)&
        (df["Age"]<=100)
    ].copy()

    return clean_df

In [10]:
cleaned_policy_data = clean_policy_data(policy_data)

cleaned_policy_data

,Policy_ID,Age,Sum_Assured,Annual_Premium,Policy_Term
0,1001,35,1000000,12000.0,20
2,1003,29,750000,NaN,25
3,1004,51,2000000,26000.0,10


In [ ]:
#final validation function

def check_negative_age(df):
    failures = df[(df["Age"]<0)]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Age",
        "Value": failures["Age"],
        "Issue": "Negative Age"
    })
    return report

def check_maximum_age(df):
    failures = df[(df["Age"]>100)]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Age",
        "Value": failures["Age"],
        "Issue": "Age Above Maximum"
    })
    return report

def check_missing_age(df):
    failures = df[(df["Age"].isnull())]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Age",
        "Value": failures["Age"],
        "Issue": "Missing Age"
    })
    return report

def check_negative_premium(df):
    failures = df[(df["Annual_Premium"]<0)]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Annual_Premium",
        "Value": failures["Annual_Premium"],
        "Issue": "Negative Premium"
    })
    return report

def check_missing_premium(df):
    failures = df[(df["Annual_Premium"].isnull())]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Annual_Premium",
        "Value": failures["Annual_Premium"],
        "Issue": "Missing Premium"
    })
    return report

def check_negative_sum_assured(df):
    failures = df[df["Sum_Assured"]<0]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Sum_Assured",
        "Value": failures["Sum_Assured"],
        "Issue": "Negative Sum Assured"
    })
    return report

def check_missing_sum_assured(df):
    failures = df[df["Sum_Assured"].isnull()]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Sum_Assured",
        "Value": failures["Sum_Assured"],
        "Issue": "Invalid Sum Assured"
    })
    return report

def check_negative_policy_term(df):
    failures = df[df["Policy_Term"]<0]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Policy_Term",
        "Value": failures["Policy_Term"],
        "Issue": "Negative Policy Term"
        })
    return report

def check_missing_policy_term(df):
    failures = df[df["Policy_Term"].isnull()]

    report = pd.DataFrame({
        "Policy_ID": failures["Policy_ID"],
        "Field": "Policy_Term",
        "Value": failures["Policy_Term"],
        "Issue": "Missing Policy Term"
        })
    return report

def validate_policy_data(df):
    report = pd.concat([
            check_negative_age(df),
            check_maximum_age(df),
            check_missing_age(df),
            check_negative_premium(df),
            check_missing_premium(df),
            check_negative_sum_assured(df),
            check_missing_sum_assured(df),
            check_negative_policy_term(df),
            check_missing_policy_term(df)
            ], ignore_index=True)
    return report

In [29]:
validation_report = validate_policy_data(policy_data)

validation_report

,Policy_ID,Field,Value,Issue
0,1002,Age,-5.0,Negative Age
1,1005,Age,138.0,Age Above Maximum
2,1003,Annual_Premium,NaN,Missing Premium


In [30]:
validation_report[
    validation_report["Field"] == "Age"
]["Policy_ID"].unique()

array([1002, 1005])

In [1]:
import pandas as pd
import numpy as np

n = 5000

df = pd.DataFrame({
    "PolicyID": range(1, n + 1),
    "ProductLine": np.random.choice(
        ["Motor", "Home", "Commercial"],
        n,
        p=[0.5, 0.3, 0.2]
    ),
    "Region": np.random.choice(
        ["North", "South", "East", "West"],
        n
    ),
    "Channel": np.random.choice(
        ["Agent", "Broker", "Direct"],
        n
    ),
    "Exposure": np.random.uniform(0.5, 1.0, n),
    "Age": np.random.randint(18, 75, n),
    "VehicleAge": np.random.randint(0, 15, n)
})

# Different risk levels by product
freq_map = {
    "Motor": 0.08,
    "Home": 0.18,
    "Commercial": 0.12
}

sev_map = {
    "Motor": 4000,
    "Home": 12000,
    "Commercial": 7000
}

df["ClaimCount"] = [
    np.random.poisson(freq_map[p])
    for p in df["ProductLine"]
]

df["ClaimAmount"] = [
    np.random.gamma(
        shape=2,
        scale=sev_map[p] / 2
    ) * max(c, 1)
    for p, c in zip(df["ProductLine"], df["ClaimCount"])
]

df["Premium"] = [
    {
        "Motor": 900,
        "Home": 1800,
        "Commercial": 1400
    }[p] * np.random.uniform(0.9, 1.1)
    for p in df["ProductLine"]
]

df.to_csv("portfolio_test_v2.csv", index=False)